In [1]:
# =============================================================================
#  HYBRID TDNN-HMM ASR — Bahasa Jawa — v6 (Perbaikan WER 500%)
#
#  PERBAIKAN DARI v5 — KRITIS (menyebabkan WER > 100%):
#
#  [FIX-v6-1] _fonem_ke_kata() fallback return [fonem_seq] →
#              Fonem diperlakukan sebagai kata oleh compute_wer() sehingga
#              menghasilkan insertion masif dan WER > 100%.
#              Sebelum: return [fonem_seq]
#              Sesudah: return []  (kosong lebih baik dari salah)
#
#  [FIX-v6-2] dekode() fallback return fonem_seq →
#              Sama seperti di atas — fonem dikembalikan sebagai output akhir
#              ketika tidak ada kandidat kata ditemukan.
#              Sebelum: if not kata_cand: return fonem_seq
#              Sesudah: if not kata_cand: return []
#
#  [FIX-v6-3] N_HMM_STATES terlalu tinggi untuk model awal →
#              Dengan threshold 4 frame, fonem pendek tersaring semua
#              sehingga output sering kosong. Diturunkan ke 2.
#              Sebelum: N_HMM_STATES = 4
#              Sesudah: N_HMM_STATES = 2
#
#  [FIX-v6-4] AVG_FRAMES_PER_WORD terlalu besar untuk kalimat pendek →
#              max_words = T // 30 membatasi pencarian kata terlalu ketat.
#              Sebelum: AVG_FRAMES_PER_WORD = 30
#              Sesudah: AVG_FRAMES_PER_WORD = 15
#
#  [FIX-v6-5] DIR_SAVE diupdate ke out_v6
#
#  PERBAIKAN DARI v4 (tetap berlaku):
#
#  [FIX-Bug-1 s/d FIX-Bug-9] — lihat catatan v5
#
# =============================================================================
#  CATATAN v5 (ringkasan):
#
#  PERBAIKAN DARI v4:
#
#  [FIX-Bug-1] normalisasi() menghapus diakritik é è â ê ô û →
#              Regex diganti agar karakter Unicode Jawa dipertahankan.
#              Sebelum: re.sub(r"[^a-z\s]","",...)
#              Sesudah: re.sub dengan allowlist Unicode diacritics
#
#  [FIX-Bug-2] Fonem "eu" dan "ae" tidak bisa diproduksi G2P →
#              Ditambahkan ke _G2P_DIGRAF sehingga G2P bisa memetakannya.
#
#  [FIX-Bug-3] dekode_semua_audio() memakai cache_mfcc_raw (pra-CMVN)
#              padahal TDNN dilatih pada cache_final (pasca fMLLR) →
#              Evaluasi sekarang menggunakan cache_final konsisten.
#
#  [FIX-Bug-4] Path scipy memakai log10, path librosa memakai log →
#              Disamakan ke natural log (np.log) di scipy path.
#
#  [FIX-Bug-5] build_vtln_filterbank() memakai VTLN_FHIGH (7200 Hz)
#              sebagai batas warp, tapi filterbank hanya sampai _FMAX
#              (4000 Hz) → vtln_warp_freq sekarang menerima f_high dari
#              parameter fungsi, bukan dari konstanta global.
#
#  [FIX-Bug-6] FMLLRTransform.fit() menerima fonem_labels tapi tidak
#              memakainya → GMM sekarang difilter berdasarkan fonem aktif
#              pada utterance jika fonem_labels tersedia.
#
#  [FIX-Bug-7] LabelSmoothingNLLLoss mengalikan class weight ke seluruh
#              distribusi smooth (non-standar) → Weight sekarang hanya
#              diterapkan pada kelas target (sesuai PyTorch NLLLoss).
#
#  [FIX-Bug-8] GaussianMixtureModel diam-diam mengurangi n_components
#              tanpa peringatan → Sekarang mencetak WARNING eksplisit.
#
#  [FIX-Bug-9] cari_vtln_alpha_per_speaker() tidak memeriksa apakah
#              ada file valid per speaker → Sekarang ada pengecekan
#              dan peringatan jika tidak ada data valid.
# =============================================================================

SAMPLED_ROOT = r"D:\doc\2026\FINAL_CODE\data\sampled_small"
CSV_PATH = (
    r"D:\doc\2026\FINAL_CODE\src\codeStat\preModelling"
    r"\samplingStratified\out_samplingStratified\dataSampledSmall.csv"
)
DIR_SAVE = (
    r"D:\doc\2026\FINAL_CODE\src\codeStat\modelling"
    r"\voiceModelling\srcVoiceModelling\hybrid\src\TDNN-HMM\out_v6"
)

COL_SPEAKER = "speaker_id"

# ── GMM ───────────────────────────────────────────────────────────────────────
N_COMPONENTS_GMM   = 8
N_ITER_GMM         = 100
TOL_GMM            = 1e-4
MIN_FRAME_GMM      = 20
GMM_EM_ITERS       = 3

# ── TDNN ──────────────────────────────────────────────────────────────────────
TDNN_HIDDEN        = 256
TDNN_DROPOUT       = 0.3
TDNN_WEIGHT_DECAY  = 1e-3
TDNN_LABEL_SMOOTH  = 0.1
TDNN_EPOCHS        = 60
TDNN_LR            = 5e-4
TDNN_LR_MIN        = 1e-6
TDNN_BATCH_SIZE    = 512
TDNN_WARMUP_EPOCH  = 5
TDNN_CONTEXTS      = [
    [-2, -1, 0, 1, 2],
    [-1,  0, 1],
    [-1,  0, 1],
    [-3,  0, 3],
    [-3,  0, 3],
    [-6,  0, 6],
]

# ── HMM / Decode ─────────────────────────────────────────────────────────────
N_HMM_STATES        = 2   # [FIX-v6-3] turun dari 4 → threshold lebih toleran
BEAM_WIDTH          = 8
AVG_FRAMES_PER_WORD = 15  # [FIX-v6-4] turun dari 30 → max_words lebih besar
MIN_FONEM_DUR       = 2
MAX_FONEM_PER_KATA  = 8

# ── LM ───────────────────────────────────────────────────────────────────────
N_GRAM         = 3
VAL_RATIO      = 0.2
SPEAKER_SPLIT  = True

# ── Triphone ─────────────────────────────────────────────────────────────────
USE_TRIPHONE       = True
TRIPHONE_MIN_FRAME = 30

# ── VTLN ─────────────────────────────────────────────────────────────────────
USE_VTLN           = True
VTLN_ALPHA_RANGE   = [0.80, 0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15, 1.20]
VTLN_FBREAK        = 4000.0   # Hz — titik patah warp piecewise linear
# [FIX-Bug-5] VTLN_FHIGH sekarang hanya dipakai sebagai default fallback;
#             build_vtln_filterbank() meneruskan f_high-nya sendiri.
VTLN_FHIGH         = 8000.0   # diselaraskan dengan Nyquist 16kHz

# ── SAT / fMLLR ──────────────────────────────────────────────────────────────
USE_SAT            = True
FMLLR_ITERS        = 3
FMLLR_MIN_FRAME    = 50
SAT_ITERS          = 2

LOG_ZERO = -1e15
LOG_EPS  = 1e-300
COV_REG  = 1e-6
FEAT_EPS = 1e-8

# =============================================================================
#  IMPORT
# =============================================================================
import os, re, glob, math, time, random, warnings, unicodedata
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    _TORCH = True
except ImportError:
    raise ImportError("PyTorch diperlukan: pip install torch")

try:
    import librosa; _LIBROSA = True
except ImportError:
    _LIBROSA = False

try:
    from scipy.io import wavfile as _wavfile; _SCIPY = True
except ImportError:
    _SCIPY = False

warnings.filterwarnings("ignore")

# =============================================================================
#  FONEM
# =============================================================================
FONEM_JAWA: List[str] = [
    "a","i","u","e","o","eu","ae",
    "p","b","t","d","k","g","dh","th",
    "m","n","ng","ny",
    "s","h","f",
    "j","c",
    "l","r",
    "w","y",
    "SIL",
]
FONEM2ID: Dict[str, int] = {f: i for i, f in enumerate(FONEM_JAWA)}
N_FONEM: int = len(FONEM_JAWA)
SIL_ID: int  = FONEM2ID["SIL"]

# [FIX-Bug-2] Tambahkan "eu" dan "ae" ke digraf agar G2P bisa memetakannya.
# Sebelumnya kedua fonem ini ada di FONEM_JAWA tapi tidak bisa diproduksi,
# menyebabkan 2 neuron output layer TDNN tidak pernah mendapat training signal.
_G2P_DIGRAF = {
    "dh": "dh",
    "th": "th",
    "ng": "ng",
    "ny": "ny",
    "eu": "eu",   # [FIX-Bug-2] ditambahkan
    "ae": "ae",   # [FIX-Bug-2] ditambahkan
}

_kf_cache: Dict[str, Tuple[str,...]] = {}
_norm_cache: Dict[str, str] = {}


def kata_ke_fonem(kata: str) -> Tuple[str,...]:
    if kata in _kf_cache: return _kf_cache[kata]
    fonem, i = [], 0
    while i < len(kata):
        if i+1 < len(kata) and kata[i:i+2] in _G2P_DIGRAF:
            fonem.append(_G2P_DIGRAF[kata[i:i+2]]); i += 2
        elif kata[i] in FONEM2ID:
            fonem.append(kata[i]); i += 1
        else:
            i += 1
    result = tuple(fonem) if fonem else ("SIL",)
    _kf_cache[kata] = result
    return result


# [FIX-Bug-1] Fungsi normalisasi diperbaiki agar TIDAK menghapus diakritik
# Jawa (é, è, â, ê, ô, û). Regex lama [^a-z\s] menghapus semua karakter
# non-ASCII sehingga "anggoné" → "anggon" (kata terpotong).
# Perbaikan: NFC normalization + allowlist Unicode diacritics.
_ALLOWED_DIACRITICS = (
    "\u00e9"  # é
    "\u00e8"  # è
    "\u00e2"  # â
    "\u00ea"  # ê
    "\u00f4"  # ô
    "\u00fb"  # û
    "\u00e0"  # à (untuk kata serapan)
    "\u00f3"  # ó (untuk kata serapan)
)
_RE_NORM = re.compile("[^a-z " + _ALLOWED_DIACRITICS + "]")

def normalisasi(teks: str) -> str:
    """
    [FIX-Bug-1] Normalisasi teks dengan mempertahankan diakritik Jawa.
    Sebelumnya: re.sub('[^a-z ]', '') menghapus e-aksen, dll.
    Sesudahnya: hanya tanda baca dan angka yang dihapus.
    """
    if teks in _norm_cache: return _norm_cache[teks]
    # NFC: satukan karakter + combining mark menjadi satu code point
    t = unicodedata.normalize("NFC", teks.lower().strip())
    # Buang tanda baca & angka, pertahankan huruf latin + diakritik Jawa
    t = _RE_NORM.sub(" ", t)
    t = re.sub(r"\s+", " ", t).strip()
    _norm_cache[teks] = t
    return t


def buat_lexicon(transkrip_list):
    lex = {}
    for kalimat in transkrip_list:
        for kata in normalisasi(kalimat).split():
            if kata not in lex:
                lex[kata] = list(kata_ke_fonem(kata))
    return lex


def _bangun_fonem_seq(label, lexicon):
    fs = []
    for kata in normalisasi(label).split():
        fs.extend(lexicon.get(kata, list(kata_ke_fonem(kata))))
    return fs

# =============================================================================
#  TRIPHONE
# =============================================================================
def buat_triphone_seq(fonem_seq):
    if not fonem_seq: return []
    p = ["SIL"] + list(fonem_seq) + ["SIL"]
    return [f"{p[i-1]}-{p[i]}+{p[i+1]}" for i in range(1, len(p)-1)]

def triphone_center(tri):
    try: return tri.split("-")[1].split("+")[0]
    except: return tri

# =============================================================================
#  CSV & SPLIT
# =============================================================================
def baca_transkrip_csv(csv_path, col_speaker=COL_SPEAKER):
    if not os.path.isfile(csv_path): raise FileNotFoundError(csv_path)
    df = None
    for enc in ("utf-8","utf-8-sig","latin-1","cp1252"):
        try: df = pd.read_csv(csv_path, encoding=enc); break
        except: continue
    if df is None: raise ValueError(f"Gagal baca CSV: {csv_path}")
    df.columns = [c.strip() for c in df.columns]
    col_path = next((c for c in ["Path_Data","path_data","path","audio_path","filename"] if c in df.columns), None)
    col_text = next((c for c in ["sentences","sentence","text","transcription","label"] if c in df.columns), None)
    if not col_path: raise ValueError(f"Kolom path tidak ada. Kolom: {list(df.columns)}")
    if not col_text: raise ValueError(f"Kolom teks tidak ada. Kolom: {list(df.columns)}")
    dt, ds = {}, {}
    has_spk = bool(col_speaker and col_speaker in df.columns)
    for _, row in df.iterrows():
        pv = str(row[col_path]).strip(); tv = str(row[col_text]).strip()
        if pv in ("nan","") or tv in ("nan",""): continue
        aid = os.path.splitext(os.path.basename(pv))[0]
        dt[aid] = tv.lower()
        if has_spk: ds[aid] = str(row[col_speaker]).strip()
    n_spk = len(set(ds.values())) if ds else 0
    print(f"  CSV: {len(dt)} audio | {n_spk} speaker unik")
    return dt, ds


def split_by_speaker(audio_ids, dict_speaker, val_ratio=VAL_RATIO):
    ids = list(audio_ids)
    if not dict_speaker:
        random.shuffle(ids); n_val = max(1, int(len(ids)*val_ratio))
        return ids[n_val:], ids[:n_val]
    spk2ids = defaultdict(list)
    for aid in ids: spk2ids[dict_speaker.get(aid,"unknown")].append(aid)
    speakers = list(spk2ids.keys()); random.shuffle(speakers)
    n_val_spk = max(1, int(len(speakers)*val_ratio))
    val_spk = set(speakers[:n_val_spk])
    train_ids, val_ids = [], []
    for spk, sids in spk2ids.items():
        (val_ids if spk in val_spk else train_ids).extend(sids)
    print(f"  Train: {len(train_ids)} | Val: {len(val_ids)}")
    return train_ids, val_ids

# =============================================================================
#  MFCC & VTLN
# =============================================================================
_SR = 16000; _N_MFCC = 13; _N_FFT = 512; _HOP_LENGTH = 160
_WIN_LENGTH = 400; _N_MELS = 26; _PRE_EMPH = 0.97
_FMIN = 0.0; _FMAX = 8000.0; _DELTA_W = 2
FEAT_DIM = _N_MFCC * 3


def vtln_warp_freq(freq_hz: float, alpha: float,
                   f_low: float = _FMIN,
                   f_high: float = VTLN_FHIGH,
                   f_break: float = VTLN_FBREAK) -> float:
    """
    VTLN piecewise-linear warping function (Wegmann & Petrone 1996).
    [FIX-Bug-5] f_high sekarang diteruskan dari parameter, bukan dari
    konstanta global VTLN_FHIGH, sehingga konsisten dengan filterbank.
    """
    if alpha == 1.0:
        return freq_hz
    f_break_warped = f_break / alpha
    if freq_hz <= f_break_warped:
        return alpha * freq_hz
    else:
        scale = (f_high - f_break) / max(f_high - f_break_warped, 1e-8)
        return f_break + (freq_hz - f_break_warped) * scale


def build_vtln_filterbank(n_mels: int, n_fft: int, sr: int, alpha: float,
                           f_low: float = _FMIN,
                           f_high: float = _FMAX) -> np.ndarray:
    """
    Bangun mel filterbank dengan VTLN warping.
    [FIX-Bug-5] vtln_warp_freq sekarang menerima f_high dari parameter
    fungsi ini (bukan dari VTLN_FHIGH global), sehingga batas atas warp
    konsisten dengan batas atas filterbank.
    Sebelumnya: vtln_warp_freq(f, alpha, f_low, VTLN_FHIGH, VTLN_FBREAK)
    Sesudahnya: vtln_warp_freq(f, alpha, f_low, f_high,     VTLN_FBREAK)
    """
    hz2mel = lambda f: 1127.0 * np.log1p(f / 700.0)
    mel2hz = lambda m: 700.0 * (np.exp(m / 1127.0) - 1.0)

    mel_pts = np.linspace(hz2mel(f_low), hz2mel(f_high), n_mels + 2)
    hz_pts  = mel2hz(mel_pts)

    # [FIX-Bug-5] Teruskan f_high dari parameter, bukan VTLN_FHIGH global
    hz_warped = np.array([
        vtln_warp_freq(f, alpha, f_low, f_high, VTLN_FBREAK)
        for f in hz_pts
    ])
    hz_warped = np.clip(hz_warped, f_low, sr / 2)

    bins = np.floor((n_fft + 1) * hz_warped / sr).astype(int)
    bins = np.clip(bins, 0, n_fft // 2)

    fb = np.zeros((n_mels, n_fft // 2 + 1), dtype=np.float32)
    for m in range(1, n_mels + 1):
        lo, mid, hi = bins[m-1], bins[m], bins[m+1]
        for k in range(lo, mid):
            fb[m-1, k] = (k - lo) / max(mid - lo, 1)
        for k in range(mid, hi):
            fb[m-1, k] = (hi - k) / max(hi - mid, 1)
    return fb


def apply_cmvn(f: np.ndarray) -> np.ndarray:
    return (f - f.mean(0, keepdims=True)) / (f.std(0, keepdims=True) + FEAT_EPS)


def apply_cmvn_speaker(cache_mfcc: dict, dict_speaker: dict) -> dict:
    if not dict_speaker:
        print("  [INFO] Tidak ada kolom speaker → skip speaker CMVN")
        return cache_mfcc

    unique_spks = set(dict_speaker.values())
    if len(unique_spks) <= 1:
        print(f"  [INFO] Hanya {len(unique_spks)} speaker → skip speaker CMVN")
        return cache_mfcc

    spk_frames = defaultdict(list)
    for aid, feat in cache_mfcc.items():
        spk_frames[dict_speaker.get(aid, "unknown")].append(feat)

    spk_mean, spk_std = {}, {}
    for spk, frames in spk_frames.items():
        all_f = np.vstack(frames)
        spk_mean[spk] = all_f.mean(0)
        spk_std[spk]  = all_f.std(0) + FEAT_EPS

    result = {}
    for aid, feat in cache_mfcc.items():
        spk = dict_speaker.get(aid, "unknown")
        result[aid] = apply_cmvn((feat - spk_mean[spk]) / spk_std[spk])
    print(f"  Speaker CMVN diterapkan: {len(unique_spks)} speaker")
    return result


def _delta_numpy(feat: np.ndarray, w: int = _DELTA_W) -> np.ndarray:
    T = feat.shape[0]; den = 2.0 * sum(n*n for n in range(1, w+1))
    pad = np.pad(feat, ((w,w),(0,0)), mode="edge"); d = np.zeros_like(feat)
    for n in range(1, w+1):
        d += n * (pad[w+n:w+n+T] - pad[w-n:w-n+T])
    return d / den


def _mfcc_librosa_alpha(wav_path: str, alpha: float = 1.0) -> np.ndarray:
    """Ekstraksi MFCC dengan VTLN alpha via librosa."""
    y, _ = librosa.load(wav_path, sr=_SR, mono=True)
    if len(y) == 0: raise ValueError(f"Audio kosong: {wav_path}")
    y = np.append(y[0], y[1:] - _PRE_EMPH * y[:-1])

    stft = np.abs(librosa.stft(y, n_fft=_N_FFT, hop_length=_HOP_LENGTH,
                                 win_length=_WIN_LENGTH, window="hamming")) ** 2

    # [FIX-Bug-5] build_vtln_filterbank dipanggil dengan f_high=_FMAX
    # sehingga batas atas warp konsisten dengan batas filterbank
    fb = build_vtln_filterbank(_N_MELS, _N_FFT, _SR, alpha, _FMIN, _FMAX)
    mel_spec = np.maximum(fb @ stft, 1e-10)

    # Gunakan natural log (np.log) — konsisten dengan scipy path [FIX-Bug-4]
    log_mel = np.log(mel_spec)

    N = _N_MELS
    dct = np.cos(np.pi * np.arange(_N_MFCC)[:, None] *
                 (np.arange(N) + 0.5)[None, :] / N)
    mfcc = (log_mel.T @ dct.T) * np.sqrt(2.0 / N)

    d1 = librosa.feature.delta(mfcc.T, width=_DELTA_W*2+1).T
    d2 = librosa.feature.delta(mfcc.T, order=2, width=_DELTA_W*2+1).T
    return np.concatenate([mfcc, d1, d2], axis=1).astype(np.float32)


def _mfcc_scipy_alpha(wav_path: str, alpha: float = 1.0) -> np.ndarray:
    """
    Ekstraksi MFCC + VTLN fallback (scipy).
    [FIX-Bug-4] Disamakan ke natural log (np.log) agar konsisten dengan
    path librosa. Sebelumnya memakai log10 yang menghasilkan skala
    berbeda ~2.303× dari path librosa.
    """
    sr, audio = _wavfile.read(wav_path)
    if audio.dtype == np.int16:   audio = audio.astype(np.float32) / 32768.0
    elif audio.dtype == np.int32: audio = audio.astype(np.float32) / 2147483648.0
    else:                          audio = audio.astype(np.float32)
    if audio.ndim > 1: audio = audio.mean(1)
    if sr != _SR:
        n_new = int(len(audio) * _SR / sr)
        audio = np.interp(np.linspace(0, len(audio)-1, n_new),
                          np.arange(len(audio)), audio)

    y = np.append(audio[0], audio[1:] - _PRE_EMPH * audio[:-1])
    nf = max(1, 1 + (len(y) - _WIN_LENGTH) // _HOP_LENGTH)
    idx = np.arange(_WIN_LENGTH)[None,:] + _HOP_LENGTH * np.arange(nf)[:,None]
    idx = np.clip(idx, 0, len(y)-1)
    fr  = y[idx].astype(np.float32) * np.hamming(_WIN_LENGTH).astype(np.float32)
    mag = np.abs(np.fft.rfft(fr, n=_N_FFT)) ** 2

    # [FIX-Bug-5] f_high=_FMAX diteruskan eksplisit
    fb = build_vtln_filterbank(_N_MELS, _N_FFT, _SR, alpha, _FMIN, _FMAX)

    # [FIX-Bug-4] Ganti log10 → np.log agar konsisten dengan librosa path
    # Sebelumnya: log_mel = np.log10(np.maximum(mag @ fb.T, 1e-10))
    log_mel = np.log(np.maximum(mag @ fb.T, 1e-10))

    N = _N_MELS
    dct = np.cos(np.pi * np.arange(_N_MFCC)[:, None] *
                 (np.arange(N) + 0.5)[None, :] / N)
    mfcc = (log_mel @ dct.T) * np.sqrt(2.0 / N)
    d1 = _delta_numpy(mfcc); d2 = _delta_numpy(d1)
    return np.concatenate([mfcc, d1, d2], axis=1).astype(np.float32)


def extract_features_alpha(wav_path: str, alpha: float = 1.0,
                            cmvn: bool = True) -> np.ndarray:
    if not os.path.isfile(wav_path): raise FileNotFoundError(wav_path)
    feat = (_mfcc_librosa_alpha(wav_path, alpha) if _LIBROSA
            else _mfcc_scipy_alpha(wav_path, alpha))
    return apply_cmvn(feat) if cmvn else feat


def build_mfcc_cache(sampled_root: str, dict_transkrip: dict,
                     alpha_map: Optional[Dict[str, float]] = None,
                     dict_speaker: Optional[Dict[str, str]] = None) -> Tuple[dict, list]:
    wf  = glob.glob(os.path.join(sampled_root, "**/*.wav"), recursive=True)
    wf += glob.glob(os.path.join(sampled_root, "**/*.flac"), recursive=True)
    cache, meta = {}, []
    print(f"  Ekstrak MFCC dari {len(wf)} file ...")
    t0 = time.time()
    for idx, wp in enumerate(wf, 1):
        aid = os.path.splitext(os.path.basename(wp))[0]
        if not dict_transkrip.get(aid, ""): continue
        try:
            alpha = 1.0
            if alpha_map and dict_speaker:
                spk = dict_speaker.get(aid, "unknown")
                alpha = alpha_map.get(spk, 1.0)
            feat = extract_features_alpha(wp, alpha=alpha, cmvn=True)
            cache[aid] = feat
            meta.append({"Id": aid, "Frames": feat.shape[0],
                          "Dim": feat.shape[1], "VTLN_alpha": alpha})
        except Exception as e:
            print(f"    [SKIP] {os.path.basename(wp)}: {e}")
        if idx % 200 == 0:
            print(f"    [{idx}/{len(wf)}] {time.time()-t0:.0f}s")
    print(f"  Selesai: {len(cache)} audio")
    return cache, meta


# =============================================================================
#  VTLN ALPHA SEARCH
# =============================================================================
def cari_vtln_alpha_per_speaker(
    wav_paths_per_speaker: Dict[str, List[str]],
    gmm_mono: Dict,
    dict_transkrip: Dict[str, str],
    lexicon: Dict,
    alpha_range: List[float] = VTLN_ALPHA_RANGE,
) -> Dict[str, float]:
    """
    Grid search VTLN alpha per speaker.
    [FIX-Bug-9] Ditambahkan pengecekan speaker tanpa data valid.
    Sebelumnya: speaker dengan 0 file valid langsung default alpha=1.0
    tanpa peringatan, menyulitkan debug.
    """
    if not gmm_mono:
        print("  [VTLN] GMM belum ada → skip search, alpha=1.0 untuk semua")
        return {spk: 1.0 for spk in wav_paths_per_speaker}

    speaker_alpha = {}
    t0 = time.time()
    print(f"  [VTLN] Grid search α ∈ {alpha_range} untuk {len(wav_paths_per_speaker)} speaker ...")

    for spk, paths in wav_paths_per_speaker.items():
        best_alpha, best_ll = 1.0, -np.inf

        for alpha in alpha_range:
            total_ll = 0.0; n_frame = 0
            for wp in paths[:10]:
                aid = os.path.splitext(os.path.basename(wp))[0]
                label = dict_transkrip.get(aid, "")
                if not label: continue
                try:
                    feat = extract_features_alpha(wp, alpha=alpha, cmvn=True)
                except Exception:
                    continue
                # [FIX-Bug-9] Cek feat kosong sebelum hitung LL
                if feat.shape[0] == 0:
                    continue
                fs = _bangun_fonem_seq(label, lexicon)
                if not fs: continue
                fonem_unik = list(set(fs))
                for fonem in fonem_unik:
                    if fonem in gmm_mono:
                        ll_vals = gmm_mono[fonem].log_prob_per_frame(feat)
                        total_ll += float(ll_vals.mean())
                        n_frame  += len(ll_vals)

            if n_frame > 0:
                avg_ll = total_ll / n_frame
                if avg_ll > best_ll:
                    best_ll = avg_ll; best_alpha = alpha

        # [FIX-Bug-9] Peringatan eksplisit jika tidak ada data valid
        if best_ll == -np.inf:
            print(f"  [VTLN WARNING] Speaker {spk}: tidak ada file valid "
                  f"→ default alpha=1.0")

        speaker_alpha[spk] = best_alpha

    n_nondefault = sum(1 for a in speaker_alpha.values() if a != 1.0)
    print(f"  [VTLN] Selesai {time.time()-t0:.1f}s | "
          f"{n_nondefault}/{len(speaker_alpha)} speaker punya α≠1.0")
    return speaker_alpha


def bangun_wav_per_speaker(sampled_root: str,
                            dict_speaker: Dict[str, str]) -> Dict[str, List[str]]:
    wf  = glob.glob(os.path.join(sampled_root, "**/*.wav"), recursive=True)
    wf += glob.glob(os.path.join(sampled_root, "**/*.flac"), recursive=True)
    spk2wav = defaultdict(list)
    for wp in wf:
        aid = os.path.splitext(os.path.basename(wp))[0]
        spk = dict_speaker.get(aid, "unknown")
        spk2wav[spk].append(wp)
    return dict(spk2wav)


# =============================================================================
#  GMM
# =============================================================================
class GaussianMixtureModel:
    def __init__(self, n_components=4, n_iter=100, tol=1e-3,
                 reg_covar=COV_REG, random_state=None):
        self.n_components = n_components; self.n_iter = n_iter
        self.tol = tol; self.reg_covar = reg_covar; self.random_state = random_state
        self.weights_ = self.means_ = self.covariances_ = None
        self.converged_ = False; self.n_iter_done_ = 0
        self.log_likelihood_history_: List[float] = []

    def _init_params(self, X):
        rng = np.random.default_rng(self.random_state); K = self.n_components
        idx = rng.choice(X.shape[0], size=K, replace=False)
        self.means_ = X[idx].copy(); self.weights_ = np.full(K, 1.0/K)
        self.covariances_ = np.tile(np.var(X, axis=0) + self.reg_covar, (K, 1))

    def _log_gaussian_all(self, X):
        mu = self.means_; var = self.covariances_
        log_det = np.sum(np.log(var), axis=1)
        diff = X[:,None,:] - mu[None,:,:]
        maha = np.sum(diff**2 / var[None,:,:], axis=2)
        return (-0.5*(X.shape[1]*np.log(2*np.pi)+log_det))[None,:] - 0.5*maha

    def _e_step(self, X):
        lw = np.log(self.weights_+LOG_EPS) + self._log_gaussian_all(X)
        lse = np.max(lw, axis=1, keepdims=True)
        ls = lse.reshape(-1) + np.log(np.sum(np.exp(lw-lse), axis=1))
        return lw - ls[:,None], ls

    def _m_step(self, X, lr):
        r = np.exp(lr); Nk = r.sum(0) + LOG_EPS
        self.weights_ = Nk / X.shape[0]; self.means_ = (r.T @ X) / Nk[:,None]
        diff = X[:,None,:] - self.means_[None,:,:]
        self.covariances_ = (np.sum(r[:,:,None]*diff**2, axis=0)/Nk[:,None]
                             + self.reg_covar)

    def fit(self, X):
        assert X.ndim == 2
        original_k = self.n_components
        self.n_components = min(self.n_components, max(1, X.shape[0]//4))

        # [FIX-Bug-8] Cetak WARNING eksplisit saat n_components dikurangi.
        # Sebelumnya pengurangan terjadi diam-diam tanpa notifikasi apapun.
        if self.n_components < original_k:
            print(f"    [WARNING] GMM: n_components {original_k}→{self.n_components} "
                  f"karena hanya {X.shape[0]} frame tersedia "
                  f"(min 4 frame/komponen)")

        self._init_params(X); self.log_likelihood_history_ = []; prev = -np.inf
        for it in range(self.n_iter):
            lr, ls = self._e_step(X); ll = float(ls.mean())
            self.log_likelihood_history_.append(ll)
            if abs(ll - prev) < self.tol:
                self.converged_ = True; self.n_iter_done_ = it+1; break
            prev = ll; self._m_step(X, lr)
        else:
            self.converged_ = False; self.n_iter_done_ = self.n_iter
        return self

    def log_prob_per_frame(self, X) -> np.ndarray:
        assert X.ndim == 2
        lw = np.log(self.weights_+LOG_EPS) + self._log_gaussian_all(X)
        lse = np.max(lw, axis=1, keepdims=True)
        return lse.reshape(-1) + np.log(np.sum(np.exp(lw-lse), axis=1))


# =============================================================================
#  SAT / fMLLR
# =============================================================================
class FMLLRTransform:
    """
    Feature-space MLLR (fMLLR / CMLLR).
    [FIX-Bug-6] fonem_labels sekarang benar-benar digunakan untuk
    memfilter GMM — hanya GMM fonem yang aktif dalam utterance yang
    digunakan untuk estimasi transform, meningkatkan akurasi fMLLR.
    """

    def __init__(self, feat_dim: int = FEAT_DIM):
        D = feat_dim
        self.D = D
        self.A = np.eye(D, dtype=np.float64)
        self.b = np.zeros(D, dtype=np.float64)

    def transform(self, X: np.ndarray) -> np.ndarray:
        return (X.astype(np.float64) @ self.A.T + self.b).astype(np.float32)

    def fit(self, X: np.ndarray, gmm_per_fonem: Dict,
            fonem_labels: Optional[np.ndarray] = None,
            n_iters: int = FMLLR_ITERS) -> "FMLLRTransform":
        """
        [FIX-Bug-6] fonem_labels sekarang digunakan untuk memfilter GMM.
        Sebelumnya parameter ini dideklarasikan tapi tidak pernah dipakai
        di dalam fungsi, sehingga semua GMM selalu digunakan tanpa
        mempertimbangkan fonem mana yang aktif di utterance ini.

        Jika fonem_labels tersedia: hanya GMM fonem yang muncul di
        alignment yang diikutkan dalam estimasi statistik sufisien.
        Jika fonem_labels None: gunakan semua GMM (perilaku lama).
        """
        if X.shape[0] < FMLLR_MIN_FRAME:
            return self

        D = self.D
        X_d = X.astype(np.float64)
        T = X_d.shape[0]

        # [FIX-Bug-6] Filter GMM berdasarkan fonem aktif di utterance ini
        if fonem_labels is not None:
            active_ids = set(int(fid) for fid in np.unique(fonem_labels)
                             if fid < len(FONEM_JAWA))
            active_names = {FONEM_JAWA[fid] for fid in active_ids}
            gmm_used = {f: g for f, g in gmm_per_fonem.items()
                        if f in active_names and g.means_ is not None}
            if not gmm_used:
                # Fallback ke semua GMM jika filter menghasilkan set kosong
                gmm_used = gmm_per_fonem
        else:
            gmm_used = gmm_per_fonem

        all_means, all_vars, all_weights = [], [], []
        for fonem, gmm in gmm_used.items():
            if gmm.means_ is None: continue
            all_means.append(gmm.means_)
            all_vars.append(gmm.covariances_)
            all_weights.append(gmm.weights_)

        if not all_means:
            return self

        means   = np.vstack(all_means)
        vars_   = np.vstack(all_vars)
        weights = np.concatenate(all_weights)
        weights = weights / (weights.sum() + LOG_EPS)

        for _ in range(n_iters):
            X_hat = X_d @ self.A.T + self.b

            log_gamma = np.log(weights[None,:] + LOG_EPS)
            diff = X_hat[:,None,:] - means[None,:,:]
            log_gamma -= 0.5 * np.sum(
                np.log(vars_[None,:,:] + COV_REG) +
                diff**2 / (vars_[None,:,:] + COV_REG),
                axis=2
            )
            log_gamma -= log_gamma.max(axis=1, keepdims=True)
            gamma = np.exp(log_gamma)
            gamma /= gamma.sum(axis=1, keepdims=True) + LOG_EPS

            X_ext = np.hstack([X_d, np.ones((T, 1))])
            W_new = np.zeros((D, D+1), dtype=np.float64)

            for i in range(D):
                inv_var_i = 1.0 / (vars_[:, i] + COV_REG)
                w_t = gamma @ inv_var_i
                G_i = (X_ext * w_t[:,None]).T @ X_ext
                G_i += np.eye(D+1) * 1e-8
                mu_weighted_i = gamma @ (means[:, i] * inv_var_i)
                k_i = (X_ext * mu_weighted_i[:,None]).sum(axis=0)
                try:
                    W_new[i] = np.linalg.solve(G_i.T, k_i)
                except np.linalg.LinAlgError:
                    W_new[i, :D] = self.A[i]
                    W_new[i, D]  = self.b[i]

            A_new = W_new[:, :D]
            b_new = W_new[:, D]
            if np.linalg.det(A_new) > 1e-10:
                self.A = A_new
                self.b = b_new

        return self


def estimasi_fmllr_per_speaker(
    cache_mfcc: Dict[str, np.ndarray],
    dict_speaker: Dict[str, str],
    gmm_per_fonem: Dict,
    alignment_cache: Optional[Dict[str, np.ndarray]] = None,
    audio_ids: Optional[List[str]] = None,
) -> Dict[str, "FMLLRTransform"]:
    if not dict_speaker:
        print("  [SAT] Tidak ada info speaker → skip fMLLR")
        return {}

    ids = audio_ids or list(cache_mfcc.keys())
    spk2feat: Dict[str, List[np.ndarray]] = defaultdict(list)
    spk2align: Dict[str, List[np.ndarray]] = defaultdict(list)

    for aid in ids:
        feat = cache_mfcc.get(aid)
        if feat is None: continue
        spk = dict_speaker.get(aid, "unknown")
        spk2feat[spk].append(feat)
        if alignment_cache and aid in alignment_cache:
            spk2align[spk].append(alignment_cache[aid])

    fmllr_per_spk: Dict[str, FMLLRTransform] = {}
    print(f"  [SAT] Estimasi fMLLR untuk {len(spk2feat)} speaker ...")
    t0 = time.time()

    for spk, feats in spk2feat.items():
        X_spk = np.vstack(feats)
        aligns = spk2align.get(spk, [])
        # [FIX-Bug-6] labels diteruskan ke fit() dan sekarang benar-benar dipakai
        labels = np.concatenate(aligns) if aligns else None

        tr = FMLLRTransform(feat_dim=X_spk.shape[1])
        tr.fit(X_spk, gmm_per_fonem, fonem_labels=labels)
        fmllr_per_spk[spk] = tr

    n_nontrivial = sum(
        1 for tr in fmllr_per_spk.values()
        if not np.allclose(tr.A, np.eye(tr.D), atol=1e-4)
    )
    print(f"  [SAT] Selesai {time.time()-t0:.1f}s | "
          f"{n_nontrivial}/{len(fmllr_per_spk)} speaker punya transform non-trivial")
    return fmllr_per_spk


def apply_fmllr_cache(
    cache_mfcc: Dict[str, np.ndarray],
    fmllr_per_spk: Dict[str, "FMLLRTransform"],
    dict_speaker: Dict[str, str],
) -> Dict[str, np.ndarray]:
    if not fmllr_per_spk:
        return cache_mfcc
    result = {}
    for aid, feat in cache_mfcc.items():
        spk = dict_speaker.get(aid, "unknown")
        tr  = fmllr_per_spk.get(spk)
        result[aid] = tr.transform(feat) if tr is not None else feat
    return result


def apply_fmllr_single(feat: np.ndarray, spk: str,
                        fmllr_per_spk: Dict[str, "FMLLRTransform"]) -> np.ndarray:
    tr = fmllr_per_spk.get(spk)
    return tr.transform(feat) if tr is not None else feat


# =============================================================================
#  ALIGNMENT — VITERBI VEKTORISASI
# =============================================================================
def alignment_fonem_viterbi(fitur: np.ndarray, fonem_seq: list,
                             gmm_per_fonem: Dict) -> np.ndarray:
    T = fitur.shape[0]; n = len(fonem_seq)
    if n == 0 or T == 0: return np.full(T, SIL_ID, dtype=np.int32)

    skor = np.full((T, n), LOG_ZERO, dtype=np.float64)
    for j, fonem in enumerate(fonem_seq):
        if fonem in gmm_per_fonem:
            skor[:, j] = gmm_per_fonem[fonem].log_prob_per_frame(fitur)
        elif fonem == "SIL":
            skor[:, j] = 0.0

    dp = np.full((T+1, n+1), LOG_ZERO, dtype=np.float64)
    bp = np.zeros((T+1, n+1), dtype=np.int8)
    dp[0, 0] = 0.0

    for t in range(1, T+1):
        e = skor[t-1]
        stay    = dp[t-1, :n] + e
        advance = np.full(n, LOG_ZERO)
        advance[1:] = dp[t-1, :n-1] + e[1:]
        use_adv = advance > stay
        dp[t, :n] = np.where(use_adv, advance, stay)
        bp[t, :n] = np.where(use_adv, 1, 0)
        dp[t, :n] = np.where(dp[t, :n] > LOG_ZERO, dp[t, :n], LOG_ZERO)

    valid = [(dp[T, j], j) for j in range(n) if dp[T, j] > LOG_ZERO]
    j_end = valid[-1][1] if valid else n-1
    path  = np.zeros(T, dtype=int); j = j_end
    for t in range(T, 0, -1):
        path[t-1] = j
        if bp[t, j] == 1: j = max(j-1, 0)

    return np.array([FONEM2ID.get(fonem_seq[k], SIL_ID) for k in path],
                    dtype=np.int32)


def alignment_triphone_viterbi(fitur: np.ndarray, fonem_seq: list,
                                gmm_tri: Dict, gmm_mono: Dict) -> np.ndarray:
    triphone_seq = buat_triphone_seq(fonem_seq)
    T = fitur.shape[0]; n = len(triphone_seq)
    if n == 0 or T == 0: return np.full(T, SIL_ID, dtype=np.int32)

    skor = np.full((T, n), LOG_ZERO, dtype=np.float64)
    for j, tri in enumerate(triphone_seq):
        if tri in gmm_tri:
            skor[:, j] = gmm_tri[tri].log_prob_per_frame(fitur)
        else:
            center = triphone_center(tri)
            if center in gmm_mono: skor[:, j] = gmm_mono[center].log_prob_per_frame(fitur)
            elif center == "SIL":  skor[:, j] = 0.0

    dp = np.full((T+1, n+1), LOG_ZERO, dtype=np.float64)
    bp = np.zeros((T+1, n+1), dtype=np.int8)
    dp[0, 0] = 0.0
    for t in range(1, T+1):
        e = skor[t-1]; stay = dp[t-1, :n] + e
        advance = np.full(n, LOG_ZERO)
        advance[1:] = dp[t-1, :n-1] + e[1:]
        use_adv = advance > stay
        dp[t, :n] = np.where(use_adv, advance, stay)
        bp[t, :n] = np.where(use_adv, 1, 0)
        dp[t, :n] = np.where(dp[t, :n] > LOG_ZERO, dp[t, :n], LOG_ZERO)

    valid  = [(dp[T, j], j) for j in range(n) if dp[T, j] > LOG_ZERO]
    j_end  = valid[-1][1] if valid else n-1
    path   = np.zeros(T, dtype=int); j = j_end
    for t in range(T, 0, -1):
        path[t-1] = j
        if bp[t, j] == 1: j = max(j-1, 0)

    centers = [triphone_center(triphone_seq[k]) for k in path]
    return np.array([FONEM2ID.get(c, SIL_ID) for c in centers], dtype=np.int32)


def kumpulkan_frame_fonem_naif(fitur: np.ndarray, fonem_seq: list) -> dict:
    T, n = fitur.shape[0], len(fonem_seq)
    if n == 0: return {}
    nf = np.ones(n, dtype=int) * (T // n)
    nf[:T - nf.sum()] += 1
    hasil = {}; idx = 0
    for i, f in enumerate(fonem_seq):
        seg = fitur[idx:idx+nf[i]]
        if seg.shape[0] > 0:
            hasil[f] = np.vstack([hasil[f], seg]) if f in hasil else seg.copy()
        idx += nf[i]
    return hasil


def kumpulkan_frame_per_fonem(cache_mfcc, dict_transkrip, lexicon,
                               gmm_mono=None, gmm_tri=None,
                               audio_ids=None) -> Dict[str, np.ndarray]:
    use_tri  = USE_TRIPHONE and gmm_tri  is not None and len(gmm_tri)  > 0
    use_mono = gmm_mono is not None and len(gmm_mono) > 0
    buf: Dict[str, List[np.ndarray]] = defaultdict(list)
    ids = audio_ids if audio_ids else list(cache_mfcc.keys())

    for audio_id in ids:
        fitur = cache_mfcc.get(audio_id); label = dict_transkrip.get(audio_id, "")
        if fitur is None or not label: continue
        fs = _bangun_fonem_seq(label, lexicon)
        if not fs: continue
        if use_tri:
            lbls = alignment_triphone_viterbi(fitur, fs, gmm_tri, gmm_mono or {})
            for fid in np.unique(lbls):
                buf[FONEM_JAWA[fid]].append(fitur[lbls == fid])
        elif use_mono:
            lbls = alignment_fonem_viterbi(fitur, fs, gmm_mono)
            for fid in np.unique(lbls):
                buf[FONEM_JAWA[fid]].append(fitur[lbls == fid])
        else:
            for fonem, arr in kumpulkan_frame_fonem_naif(fitur, fs).items():
                buf[fonem].append(arr)

    return {f: np.vstack(parts) for f, parts in buf.items()}


def kumpulkan_frame_per_triphone(cache_mfcc, dict_transkrip, lexicon,
                                  gmm_mono, audio_ids=None) -> Dict[str, np.ndarray]:
    buf: Dict[str, List[np.ndarray]] = defaultdict(list)
    ids = audio_ids if audio_ids else list(cache_mfcc.keys())
    for audio_id in ids:
        fitur = cache_mfcc.get(audio_id); label = dict_transkrip.get(audio_id, "")
        if fitur is None or not label: continue
        fs   = _bangun_fonem_seq(label, lexicon)
        if not fs: continue
        tris = buat_triphone_seq(fs); T = fitur.shape[0]
        n_tri = len(tris); seg_len = max(1, T // n_tri) if n_tri > 0 else T
        idx = 0
        for k, tri in enumerate(tris):
            end = idx + seg_len if k < n_tri-1 else T
            seg = fitur[idx:end]
            if seg.shape[0] > 0: buf[tri].append(seg)
            idx = end
    return {tri: np.vstack(parts) for tri, parts in buf.items()
            if np.vstack(parts).shape[0] >= TRIPHONE_MIN_FRAME}


def latih_gmm_per_fonem(frame_per_fonem, n_components=N_COMPONENTS_GMM,
                         n_iter=N_ITER_GMM, tol=TOL_GMM, min_frame=MIN_FRAME_GMM,
                         verbose=True) -> Dict:
    gmm = {}; t0 = time.time()
    for i, (fonem, X) in enumerate(sorted(frame_per_fonem.items())):
        if X.shape[0] < min_frame:
            if verbose: print(f"    SKIP {fonem}: {X.shape[0]} frame")
            continue
        g = GaussianMixtureModel(n_components=n_components, n_iter=n_iter, tol=tol,
                                  reg_covar=COV_REG,
                                  random_state=hash(fonem) % 100_000)
        g.fit(X); gmm[fonem] = g
        if verbose:
            ll = round(g.log_likelihood_history_[-1], 3)
            print(f"    [{i+1:2d}] {fonem:4s}: {X.shape[0]:6d} frame, "
                  f"K={g.n_components}, iter={g.n_iter_done_}, "
                  f"conv={g.converged_}, LL={ll}")
    print(f"  GMM: {len(gmm)} model dalam {time.time()-t0:.1f}s")
    return gmm


def latih_gmm_iteratif(
    cache_train: Dict[str, np.ndarray],
    dict_transkrip: Dict[str, str],
    lexicon: Dict,
    n_iters: int = GMM_EM_ITERS,
) -> Tuple[Dict, Dict, Dict[str, np.ndarray]]:
    gmm_mono = {}; frame_per_fonem = {}
    print("="*60); print("FASE 1: Iterative monophone GMM")
    for it in range(n_iters):
        print(f"\n  Iter {it+1}/{n_iters} ...")
        frame_per_fonem = kumpulkan_frame_per_fonem(
            cache_train, dict_transkrip, lexicon,
            gmm_mono=gmm_mono if it > 0 else None
        )
        gmm_mono = latih_gmm_per_fonem(frame_per_fonem,
                                        verbose=(it == n_iters-1))
        print(f"  → {len(gmm_mono)} model monophone")

    gmm_tri = {}
    if USE_TRIPHONE:
        print("\n" + "="*60); print("FASE 2: Triphone GMM")
        fpt = kumpulkan_frame_per_triphone(cache_train, dict_transkrip,
                                            lexicon, gmm_mono)
        gmm_tri = latih_gmm_per_fonem(
            fpt, n_components=max(2, N_COMPONENTS_GMM//2),
            min_frame=TRIPHONE_MIN_FRAME, verbose=False
        )
        print(f"  → {len(gmm_tri)} model triphone")

    return gmm_mono, gmm_tri, frame_per_fonem


def hitung_alignment_corpus(audio_ids, cache_mfcc, dict_transkrip,
                             lexicon, gmm_mono, gmm_tri=None) -> Dict[str, np.ndarray]:
    use_tri = USE_TRIPHONE and gmm_tri is not None and len(gmm_tri) > 0
    ac = {}
    for aid in audio_ids:
        fitur = cache_mfcc.get(aid); label = dict_transkrip.get(aid, "")
        if fitur is None or not label: continue
        fs = _bangun_fonem_seq(label, lexicon)
        if not fs: continue
        if use_tri:
            ac[aid] = alignment_triphone_viterbi(fitur, fs, gmm_tri, gmm_mono)
        else:
            ac[aid] = alignment_fonem_viterbi(fitur, fs, gmm_mono)
    return ac


# =============================================================================
#  DATASET
# =============================================================================
class FrameLevelDataset(Dataset):
    def __init__(self, audio_ids, cache_mfcc, alignment_cache):
        fp, lp = [], []
        for aid in audio_ids:
            fitur  = cache_mfcc.get(aid)
            aligns = alignment_cache.get(aid)
            if fitur is None or aligns is None: continue
            fp.append(fitur); lp.append(aligns)
        if fp:
            self.frames = torch.from_numpy(np.vstack(fp))
            self.labels = torch.from_numpy(
                np.concatenate(lp).astype(np.int64))
        else:
            self.frames = torch.zeros((0, FEAT_DIM), dtype=torch.float32)
            self.labels = torch.zeros(0, dtype=torch.long)
        print(f"  Dataset: {len(self.frames):,} frame")

    def __len__(self): return len(self.frames)
    def __getitem__(self, i): return self.frames[i], self.labels[i]


# =============================================================================
#  TDNN
# =============================================================================
class TDNNLayer(nn.Module):
    def __init__(self, input_dim, output_dim, context, dropout=0.0):
        super().__init__()
        self.context = sorted(context)
        self.linear  = nn.Linear(input_dim * len(context), output_dim)
        self.bn      = nn.BatchNorm1d(output_dim)
        self.drop    = nn.Dropout(dropout)

    def forward(self, x):
        sq = x.dim() == 2
        if sq: x = x.unsqueeze(1)
        B, T, _ = x.shape
        cmin = min(self.context); cmax = max(self.context)
        xp = F.pad(x.permute(0,2,1),
                   (max(0,-cmin), max(0,cmax))).permute(0,2,1)
        ctx = [xp[:, c-cmin:c-cmin+T, :] for c in self.context]
        sp  = torch.cat(ctx, dim=-1)
        out = self.linear(sp)
        out = self.bn(out.permute(0,2,1)).permute(0,2,1)
        out = F.relu(out); out = self.drop(out)
        return out.squeeze(1) if sq else out


class TDNNAcousticModel(nn.Module):
    def __init__(self, input_dim=FEAT_DIM, n_classes=N_FONEM,
                 hidden_dim=TDNN_HIDDEN, contexts=None, dropout=TDNN_DROPOUT):
        super().__init__()
        if contexts is None: contexts = TDNN_CONTEXTS
        layers = []; in_dim = input_dim
        for ctx in contexts:
            layers.append(TDNNLayer(in_dim, hidden_dim, ctx, dropout))
            in_dim = hidden_dim
        self.tdnn_layers = nn.ModuleList(layers)
        self.classifier  = nn.Linear(hidden_dim, n_classes)

    def forward(self, x):
        for layer in self.tdnn_layers: x = layer(x)
        return F.log_softmax(self.classifier(x), dim=-1)

    def get_posteriors_sequence(self, x):
        with torch.no_grad(): return self.forward(x)[0]


# =============================================================================
#  TRAINING
# =============================================================================
class LabelSmoothingNLLLoss(nn.Module):
    """
    [FIX-Bug-7] Class weight sekarang diterapkan hanya pada kelas target,
    bukan pada seluruh distribusi smooth.
    Sebelumnya: loss = -(smooth_dist * log_probs) * weight  (semua kelas)
    Sesudahnya: loss_per_sample * weight[target]  (hanya kelas target)

    Ini sesuai dengan perilaku standar PyTorch nn.NLLLoss(weight=...) di mana
    weight hanya mengatur kontribusi loss dari sample dengan label tertentu.
    """
    def __init__(self, n_classes, smoothing=0.1, weight=None):
        super().__init__()
        self.smoothing = smoothing
        self.n_classes = n_classes
        self.weight = weight

    def forward(self, log_probs, targets):
        with torch.no_grad():
            smooth_val  = self.smoothing / self.n_classes
            smooth_dist = torch.full_like(log_probs, smooth_val)
            smooth_dist.scatter_(1, targets.unsqueeze(1),
                                 1.0 - self.smoothing + smooth_val)

        # Hitung loss per sample (sum atas kelas)
        loss_per_sample = -(smooth_dist * log_probs).sum(dim=1)

        # [FIX-Bug-7] Terapkan weight berdasarkan kelas TARGET saja,
        # bukan dikalikan ke seluruh matriks distribusi
        if self.weight is not None:
            target_weight = self.weight[targets]   # (B,) — weight kelas target
            loss_per_sample = loss_per_sample * target_weight

        return loss_per_sample.mean()


def latih_tdnn_frame_level(
    train_ids, val_ids, cache_mfcc, alignment_train, alignment_val,
    dir_save, epochs=TDNN_EPOCHS, lr=TDNN_LR, batch_size=TDNN_BATCH_SIZE,
    hidden_dim=TDNN_HIDDEN, contexts=None, dropout=TDNN_DROPOUT,
    warmup_epochs=TDNN_WARMUP_EPOCH,
) -> TDNNAcousticModel:
    if contexts is None: contexts = TDNN_CONTEXTS
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_path = os.path.join(dir_save, "model_tdnn_fonem_v6.pt")
    os.makedirs(dir_save, exist_ok=True)

    model = TDNNAcousticModel(input_dim=FEAT_DIM, n_classes=N_FONEM,
                               hidden_dim=hidden_dim, contexts=contexts,
                               dropout=dropout).to(device)

    if os.path.isfile(model_path):
        try:
            model.load_state_dict(torch.load(model_path, map_location=device))
            print(f"  Checkpoint dimuat: {model_path}"); return model
        except Exception:
            os.remove(model_path)

    print("  Membangun dataset ...")
    train_ds = FrameLevelDataset(train_ids, cache_mfcc, alignment_train)
    val_ds   = FrameLevelDataset(val_ids,   cache_mfcc, alignment_val)
    if len(train_ds) == 0:
        print("  [ERROR] Train dataset kosong!"); return model

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               num_workers=0, pin_memory=False)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                               num_workers=0, pin_memory=False)

    label_counts = torch.bincount(train_ds.labels, minlength=N_FONEM).float()
    label_counts = torch.clamp(label_counts, min=1.0)
    weights = (1.0 / torch.sqrt(label_counts))
    weights = (weights / weights.sum()).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr,
                                   weight_decay=TDNN_WEIGHT_DECAY)

    def lr_lambda(ep):
        return float(ep+1) / float(warmup_epochs) if ep < warmup_epochs else 1.0

    warmup_sched = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(epochs - warmup_epochs, 1), eta_min=TDNN_LR_MIN)

    # [FIX-Bug-7] Loss function dengan weight yang diperbaiki
    loss_fn = LabelSmoothingNLLLoss(N_FONEM,
                                     smoothing=TDNN_LABEL_SMOOTH,
                                     weight=weights)

    riwayat = []; best_val = float("inf"); t0 = time.time()
    print(f"\n  Device: {device} | Train: {len(train_ds):,} | Val: {len(val_ds):,}")

    for epoch in range(1, epochs+1):
        model.train(); tl = nc = nt = 0
        for feats, lbls in train_loader:
            feats = feats.to(device); lbls = lbls.to(device)
            optimizer.zero_grad()
            lp = model(feats); loss = loss_fn(lp, lbls)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            tl += loss.item()*len(lbls)
            nc += (lp.argmax(1)==lbls).sum().item()
            nt += len(lbls)

        model.eval(); vl = vc = vt = 0
        with torch.no_grad():
            for feats, lbls in val_loader:
                feats = feats.to(device); lbls = lbls.to(device)
                lp = model(feats)
                vl += loss_fn(lp, lbls).item()*len(lbls)
                vc += (lp.argmax(1)==lbls).sum().item()
                vt += len(lbls)

        if epoch <= warmup_epochs: warmup_sched.step()
        else: cosine_sched.step()

        atr = tl/max(nt,1); avl = vl/max(vt,1)
        actr = 100*nc/max(nt,1); acvl = 100*vc/max(vt,1)
        cur_lr = optimizer.param_groups[0]['lr']
        riwayat.append({"Epoch": epoch, "LR": round(cur_lr,7),
                         "Train Loss": round(atr,5), "Val Loss": round(avl,5),
                         "Train Acc (%)": round(actr,2),
                         "Val Acc (%)": round(acvl,2)})
        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{epochs} | "
                  f"Loss {atr:.4f}/{avl:.4f} | "
                  f"Acc {actr:.1f}/{acvl:.1f}% | LR {cur_lr:.2e}")

        if avl < best_val:
            best_val = avl
            torch.save(model.state_dict(), model_path)

    if os.path.isfile(model_path):
        model.load_state_dict(torch.load(model_path, map_location=device))
        print(f"  Best model dimuat (val_loss={best_val:.5f})")

    pd.DataFrame(riwayat).to_csv(
        os.path.join(dir_save, "HasilTrainingTDNN_v6.csv"), index=False)
    return model


# =============================================================================
#  DECODER
# =============================================================================
class TDNNHMMDecoder:
    """
    [FIX-Bug-3] Decoder sekarang menerima cache yang sudah dinormalisasi
    (post-CMVN + post-fMLLR) agar konsisten dengan fitur yang dipakai
    saat training TDNN. Test-time fMLLR adaptation tetap tersedia untuk
    speaker yang tidak ada di training.
    """
    def __init__(self, model, lexicon, ngram_model, gmm_per_fonem,
                 device=None, n_hmm_states=N_HMM_STATES, beam_width=BEAM_WIDTH,
                 fmllr_per_spk: Optional[Dict] = None,
                 dict_speaker: Optional[Dict] = None):
        self.model = model; self.lexicon = lexicon; self.ngram = ngram_model
        self.gmm_per_fonem = gmm_per_fonem
        self.device = device or torch.device("cpu")
        self.n_hmm_states = n_hmm_states; self.beam_width = beam_width
        self.fmllr_per_spk = fmllr_per_spk or {}
        self.dict_speaker   = dict_speaker  or {}

        self.rev_lex: Dict[Tuple, List[str]] = defaultdict(list)
        for kata, fonems in lexicon.items():
            self.rev_lex[tuple(fonems)].append(kata)
        lengths = [len(v) for v in lexicon.values()]
        self.avg_fpw = float(np.mean(lengths)) if lengths else 3.0
        self.model.to(self.device); self.model.eval()

    def _adapt_test(self, fitur: np.ndarray, audio_id: str) -> np.ndarray:
        """
        [FIX-Bug-3] Test-time adaptation hanya untuk speaker yang TIDAK ada
        di training (fmllr_per_spk tidak punya entry untuk speaker ini).
        Untuk speaker training, fitur sudah di-transform sebelum masuk
        ke dekode_semua_audio() sehingga tidak perlu re-transform di sini.
        """
        if not self.fmllr_per_spk:
            return fitur
        spk = self.dict_speaker.get(audio_id, None)
        if spk and spk in self.fmllr_per_spk:
            # Speaker training: transform sudah diterapkan di cache_final
            # Tidak perlu transform ulang
            return fitur
        # Speaker baru (tidak ada di training): estimasi fMLLR dari utterance
        if fitur.shape[0] >= FMLLR_MIN_FRAME:
            tr = FMLLRTransform(feat_dim=fitur.shape[1])
            tr.fit(fitur, self.gmm_per_fonem, n_iters=1)
            return tr.transform(fitur)
        return fitur

    def _get_posteriors(self, fitur: np.ndarray) -> np.ndarray:
        t = torch.from_numpy(fitur).unsqueeze(0).to(self.device)
        return self.model.get_posteriors_sequence(t).cpu().numpy()

    def _ctc_collapse(self, fonem_ids: np.ndarray) -> List[str]:
        result = []; i = 0
        while i < len(fonem_ids):
            fid = fonem_ids[i]; j = i+1
            while j < len(fonem_ids) and fonem_ids[j] == fid: j += 1
            dur = j - i
            if dur >= self.n_hmm_states and fid != SIL_ID:
                result.append(FONEM_JAWA[fid])
            i = j
        return result

    def _fonem_ke_kata(self, fonem_seq: list, T: int) -> list:
        n = len(fonem_seq)
        if n == 0: return [[]]
        max_words = max(1, T // AVG_FRAMES_PER_WORD)
        dp = [[] for _ in range(n+1)]; dp[0] = [(0.0, [])]
        for i in range(n):
            if not dp[i]: continue
            for j in range(i+1, min(n+1, i+MAX_FONEM_PER_KATA+1)):
                sub = tuple(fonem_seq[i:j])
                if sub not in self.rev_lex: continue
                for kata in self.rev_lex[sub]:
                    for score, words in dp[i]:
                        if len(words) >= max_words: continue
                        new_words = words + [kata]
                        new_score = score + self.ngram.log_prob(
                            kata, tuple(words[-(self.ngram.n-1):]))
                        dp[j].append((new_score, new_words))
                if len(dp[j]) > self.beam_width * 3:
                    dp[j].sort(key=lambda x: x[0], reverse=True)
                    dp[j] = dp[j][:self.beam_width]
        if dp[n]:
            dp[n].sort(key=lambda x: x[0], reverse=True)
            return [w for _, w in dp[n][:self.beam_width]]
        # [FIX-v6-1] Jangan return fonem sebagai kata — insertion masif → WER 500%
        # Sebelumnya: return [fonem_seq]  ← fonem dihitung sebagai kata oleh WER
        return []

    def dekode(self, fitur: np.ndarray, audio_id: str = "") -> list:
        if fitur.shape[0] == 0: return []
        fitur_adapted = self._adapt_test(fitur, audio_id)
        log_post  = self._get_posteriors(fitur_adapted)
        fonem_ids = np.argmax(log_post, axis=1)
        fonem_seq = self._ctc_collapse(fonem_ids)
        if not fonem_seq: return []
        kata_cand = self._fonem_ke_kata(fonem_seq, fitur.shape[0])
        # [FIX-v6-2] Jangan return fonem_seq jika tidak ada kata ditemukan.
        # Sebelumnya: if not kata_cand: return fonem_seq
        # fonem_seq adalah list string fonem (bukan kata) — ketika dihitung
        # oleh compute_wer() sebagai token, setiap fonem jadi "kata" palsu
        # menghasilkan puluhan insertion per kalimat → WER meledak ke 500%.
        if not kata_cand: return []
        return self.ngram.rescore(kata_cand)


# =============================================================================
#  N-GRAM LM
# =============================================================================
class NGramModel:
    def __init__(self, n=3, discount=0.75):
        self.n = n; self.discount = discount
        self.ngram_count    = defaultdict(int)
        self.context_count  = defaultdict(int)
        self.unigram_count  = defaultdict(int)
        self.vocab          = set()
        self.N_total        = 0
        self._ctx_div       = {}

    def _tok(self, s):
        t = normalisasi(s).split()
        return ["<s>"] * (self.n-1) + t + ["</s>"]

    def fit(self, korpus):
        cw = defaultdict(set)
        for s in korpus:
            t = self._tok(s); self.vocab.update(t)
            self.N_total += len(t)
            for tok in t: self.unigram_count[tok] += 1
            for i in range(len(t) - self.n + 1):
                ng = tuple(t[i:i+self.n]); c = ng[:-1]
                self.ngram_count[ng] += 1
                self.context_count[c] += 1
                cw[c].add(ng[-1])
        self._ctx_div = {c: len(w) for c, w in cw.items()}
        return self

    def log_prob(self, kata, ctx):
        ng  = ctx + (kata,); Cng = self.ngram_count.get(ng, 0)
        Cc  = self.context_count.get(ctx, 0)
        Cu  = self.unigram_count.get(kata, 0)
        puni = (Cu + 1) / (self.N_total + len(self.vocab))
        if Cc == 0: return math.log(puni + LOG_EPS)
        lam = self.discount * self._ctx_div.get(ctx, 0) / Cc
        return math.log(max(Cng - self.discount, 0) / Cc + lam*puni + LOG_EPS)

    def score_sequence(self, words):
        t = ["<s>"] * (self.n-1) + words + ["</s>"]; lp = 0.0
        for i in range(self.n-1, len(t)):
            lp += self.log_prob(t[i], tuple(t[i-self.n+1:i]))
        return lp

    def rescore(self, kandidat):
        return max(kandidat, key=self.score_sequence) if kandidat else []


# =============================================================================
#  WER
# =============================================================================
def compute_wer(ref, hyp):
    R, H = len(ref), len(hyp)
    dp = [[0]*(H+1) for _ in range(R+1)]
    ops = [[""]*(H+1) for _ in range(R+1)]
    for i in range(R+1): dp[i][0] = i; ops[i][0] = "D"*i
    for j in range(H+1): dp[0][j] = j; ops[0][j] = "I"*j
    for i in range(1, R+1):
        for j in range(1, H+1):
            if ref[i-1] == hyp[j-1]:
                dp[i][j] = dp[i-1][j-1]; ops[i][j] = "M"
            else:
                sub = dp[i-1][j-1]+1; dlt = dp[i-1][j]+1; ins = dp[i][j-1]+1
                best = min(sub, dlt, ins); dp[i][j] = best
                ops[i][j] = ("S" if best==sub else
                              ("D" if best==dlt else "I"))
    S = D = I = 0; i, j = R, H
    while i > 0 or j > 0:
        op = ops[i][j]
        if   op == "M": i-=1; j-=1
        elif op == "S": S+=1; i-=1; j-=1
        elif op == "D": D+=1; i-=1
        elif op == "I": I+=1; j-=1
        else:
            if i > 0: D+=1; i-=1
            else:     I+=1; j-=1
    return {"wer_persen": round((S+D+I)/R*100, 2) if R > 0 else 0.0,
            "S": S, "D": D, "I": I, "N": R}


def dekode_semua_audio(cache_mfcc, audio_ids, dict_transkrip, decoder,
                        verbose=True):
    """
    [FIX-Bug-3] Fungsi ini sekarang menerima cache yang sudah melalui
    transformasi lengkap (CMVN + VTLN + fMLLR). Di main(), dipanggil
    dengan cache_final, bukan cache_mfcc_raw.
    """
    hasil = []; t0 = time.time()
    for idx, aid in enumerate(audio_ids, 1):
        fitur = cache_mfcc.get(aid); label = dict_transkrip.get(aid, "")
        if fitur is None or not label: continue
        ref = normalisasi(label).split()
        if not ref: continue
        try:
            hyp = decoder.dekode(fitur, audio_id=aid)
        except Exception as e:
            if verbose: print(f"    [ERROR] {aid}: {e}"); continue
        wr = compute_wer(ref, hyp)
        hasil.append({"Id": aid,
                       "Referensi": " ".join(ref),
                       "Hipotesis": " ".join(hyp),
                       "WER (%)": wr["wer_persen"],
                       "S": wr["S"], "D": wr["D"],
                       "I": wr["I"], "N": wr["N"]})
        if verbose and idx % 100 == 0:
            print(f"    [{idx}/{len(audio_ids)}] {time.time()-t0:.1f}s")
    return hasil


def ringkasan_wer(df: pd.DataFrame, label: str = "") -> dict:
    if df.empty: return {}
    S = int(df.S.sum()); D = int(df.D.sum())
    I = int(df.I.sum()); N = int(df.N.sum())
    wc = (S+D+I)/N*100 if N > 0 else 100.0
    print(f"\n  === WER: {label} ===")
    print(f"  Audio: {len(df)} | WER corpus: {wc:.2f}% | "
          f"Mean: {df['WER (%)'].mean():.2f}%")
    print(f"  S={S} D={D} I={I} N={N}")
    return {"wer_corpus": round(wc,2),
            "wer_mean":   round(df['WER (%)'].mean(),2),
            "wer_median": round(df['WER (%)'].median(),2),
            "wer_std":    round(df['WER (%)'].std(),2),
            "total_S": S, "total_D": D, "total_I": I, "total_N": N}


# =============================================================================
#  MAIN
# =============================================================================
def main():
    os.makedirs(DIR_SAVE, exist_ok=True)
    random.seed(42); np.random.seed(42)
    t_start = time.time()

    # ── [1] Load data ─────────────────────────────────────────────────────────
    print("\n[1/10] Membaca CSV ...")
    dict_transkrip, dict_speaker = baca_transkrip_csv(CSV_PATH)

    # ── [2] MFCC awal (alpha=1.0) ─────────────────────────────────────────────
    print("\n[2/10] Ekstraksi MFCC awal (alpha=1.0) ...")
    cache_mfcc_raw, meta = build_mfcc_cache(SAMPLED_ROOT, dict_transkrip)
    pd.DataFrame(meta).to_csv(f"{DIR_SAVE}/HasilMFCC_raw.csv", index=False)

    cache_mfcc = apply_cmvn_speaker(cache_mfcc_raw, dict_speaker)

    # ── [3] Split ──────────────────────────────────────────────────────────────
    print("\n[3/10] Split train/val ...")
    all_ids = list(cache_mfcc.keys())
    if SPEAKER_SPLIT and dict_speaker:
        train_ids, val_ids = split_by_speaker(all_ids, dict_speaker, VAL_RATIO)
    else:
        random.shuffle(all_ids); n_val = max(1, int(len(all_ids)*VAL_RATIO))
        train_ids, val_ids = all_ids[n_val:], all_ids[:n_val]
        print(f"  Train: {len(train_ids)} | Val: {len(val_ids)}")

    # ── [4] Lexicon + LM ──────────────────────────────────────────────────────
    print("\n[4/10] Lexicon dan N-gram LM ...")
    semua_transkrip = list(dict_transkrip.values())
    lexicon = buat_lexicon(semua_transkrip)
    pd.DataFrame([{"Kata": k, "Fonem": " ".join(v)}
                  for k, v in lexicon.items()]
                 ).to_csv(f"{DIR_SAVE}/HasilLexicon.csv", index=False)
    lm = NGramModel(n=N_GRAM)
    lm.fit([dict_transkrip[a] for a in train_ids if a in dict_transkrip])
    print(f"  Lexicon: {len(lexicon)} kata | LM vocab: {len(lm.vocab)}")

    # ── [5] GMM awal ──────────────────────────────────────────────────────────
    print("\n[5/10] GMM awal ...")
    cache_train_raw = {k: cache_mfcc[k] for k in train_ids if k in cache_mfcc}
    gmm_mono, gmm_tri, frame_per_fonem = latih_gmm_iteratif(
        cache_train_raw, dict_transkrip, lexicon)

    gmm_stat = [
        {"Fonem": f,
         "Frames": frame_per_fonem.get(f, np.empty((0,1))).shape[0],
         "K": g.n_components, "Iter": g.n_iter_done_,
         "Conv": g.converged_,
         "LL": round(g.log_likelihood_history_[-1], 4)
               if g.log_likelihood_history_ else None}
        for f, g in gmm_mono.items()
    ]
    pd.DataFrame(gmm_stat).to_csv(f"{DIR_SAVE}/HasilGMM_raw.csv", index=False)

    # ── [6] VTLN ──────────────────────────────────────────────────────────────
    speaker_alpha_map = {}
    if USE_VTLN and dict_speaker:
        print("\n[6/10] VTLN: grid search alpha per speaker ...")
        wav_per_spk = bangun_wav_per_speaker(SAMPLED_ROOT, dict_speaker)
        # [FIX-Bug-9] Warning untuk speaker tanpa data valid ada di dalam fungsi
        speaker_alpha_map = cari_vtln_alpha_per_speaker(
            wav_per_spk, gmm_mono, dict_transkrip, lexicon, VTLN_ALPHA_RANGE)
        pd.DataFrame([{"Speaker": spk, "VTLN_alpha": alpha}
                      for spk, alpha in speaker_alpha_map.items()]
                     ).to_csv(f"{DIR_SAVE}/HasilVTLN_alpha.csv", index=False)
        print("  Rebuild MFCC cache dengan VTLN alpha ...")
        cache_mfcc_raw, meta_vtln = build_mfcc_cache(
            SAMPLED_ROOT, dict_transkrip, speaker_alpha_map, dict_speaker)
        cache_mfcc = apply_cmvn_speaker(cache_mfcc_raw, dict_speaker)
        pd.DataFrame(meta_vtln).to_csv(
            f"{DIR_SAVE}/HasilMFCC_VTLN.csv", index=False)
    else:
        print("\n[6/10] VTLN dinonaktifkan — skip")

    # ── [7] SAT ───────────────────────────────────────────────────────────────
    cache_mfcc_sat = cache_mfcc
    fmllr_per_spk  = {}
    gmm_mono_sat   = gmm_mono
    gmm_tri_sat    = gmm_tri

    if USE_SAT and dict_speaker:
        print(f"\n[7/10] SAT: {SAT_ITERS} iterasi fMLLR + GMM ...")
        for sat_it in range(1, SAT_ITERS + 1):
            print(f"\n  === SAT iterasi {sat_it}/{SAT_ITERS} ===")
            align_sat = hitung_alignment_corpus(
                train_ids, cache_mfcc_sat, dict_transkrip, lexicon,
                gmm_mono_sat, gmm_tri_sat)
            # [FIX-Bug-6] alignment diteruskan dan benar-benar dipakai di fit()
            fmllr_per_spk = estimasi_fmllr_per_speaker(
                cache_mfcc_sat, dict_speaker, gmm_mono_sat,
                alignment_cache=align_sat, audio_ids=train_ids)
            cache_mfcc_sat = apply_fmllr_cache(
                cache_mfcc, fmllr_per_spk, dict_speaker)
            cache_train_sat = {k: cache_mfcc_sat[k]
                               for k in train_ids if k in cache_mfcc_sat}
            gmm_mono_sat, gmm_tri_sat, frame_per_fonem = latih_gmm_iteratif(
                cache_train_sat, dict_transkrip, lexicon, n_iters=2)
            print(f"  SAT iter {sat_it}: GMM diperbarui "
                  f"({len(gmm_mono_sat)} mono, {len(gmm_tri_sat)} tri)")

        gmm_stat_sat = [
            {"Fonem": f,
             "Frames": frame_per_fonem.get(f, np.empty((0,1))).shape[0],
             "K": g.n_components, "Iter": g.n_iter_done_,
             "Conv": g.converged_,
             "LL": round(g.log_likelihood_history_[-1], 4)
                   if g.log_likelihood_history_ else None}
            for f, g in gmm_mono_sat.items()
        ]
        pd.DataFrame(gmm_stat_sat).to_csv(
            f"{DIR_SAVE}/HasilGMM_SAT.csv", index=False)
        fmllr_records = [
            {"Speaker": spk,
             "DetA": float(np.linalg.det(tr.A)),
             "NormB": float(np.linalg.norm(tr.b))}
            for spk, tr in fmllr_per_spk.items()
        ]
        pd.DataFrame(fmllr_records).to_csv(
            f"{DIR_SAVE}/HasilfMLLR_params.csv", index=False)
    else:
        print("\n[7/10] SAT dinonaktifkan — skip")

    cache_final    = cache_mfcc_sat
    gmm_mono_final = gmm_mono_sat
    gmm_tri_final  = gmm_tri_sat

    # ── [8] Alignment akhir ───────────────────────────────────────────────────
    print("\n[8/10] Alignment akhir pada fitur final ...")
    align_train = hitung_alignment_corpus(
        train_ids, cache_final, dict_transkrip, lexicon,
        gmm_mono_final, gmm_tri_final)
    align_val   = hitung_alignment_corpus(
        val_ids, cache_final, dict_transkrip, lexicon,
        gmm_mono_final, gmm_tri_final)
    print(f"  Train: {len(align_train)} | Val: {len(align_val)}")

    # ── [9] TDNN ──────────────────────────────────────────────────────────────
    print("\n[9/10] Training TDNN pada fitur fMLLR-SAT ...")
    model = latih_tdnn_frame_level(
        train_ids, val_ids, cache_final,
        align_train, align_val, DIR_SAVE)

    # ── [10] Decode + evaluasi ────────────────────────────────────────────────
    print("\n[10/10] Dekode dan evaluasi ...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    decoder = TDNNHMMDecoder(
        model=model, lexicon=lexicon, ngram_model=lm,
        gmm_per_fonem=gmm_mono_final, device=device,
        n_hmm_states=N_HMM_STATES, beam_width=BEAM_WIDTH,
        fmllr_per_spk=fmllr_per_spk,
        dict_speaker=dict_speaker,
    )

    # [FIX-Bug-3] Gunakan cache_final (bukan cache_mfcc_raw) untuk evaluasi
    # agar domain fitur konsisten dengan yang dipakai saat training TDNN.
    print("  Dekode train ...")
    hasil_train = dekode_semua_audio(
        cache_final, train_ids, dict_transkrip, decoder)
    print("  Dekode val ...")
    hasil_val   = dekode_semua_audio(
        cache_final, val_ids,   dict_transkrip, decoder)

    df_tr = pd.DataFrame(hasil_train); df_vl = pd.DataFrame(hasil_val)
    df_tr.to_csv(f"{DIR_SAVE}/HasilWER_Train.csv", index=False)
    df_vl.to_csv(f"{DIR_SAVE}/HasilWER_Val.csv",   index=False)

    r_tr = ringkasan_wer(df_tr, "Train")
    r_vl = ringkasan_wer(df_vl, "Val")
    pd.DataFrame([{"set": "train", **r_tr}, {"set": "val", **r_vl}]
                 ).to_csv(f"{DIR_SAVE}/RingkasanWER.csv", index=False)

    if speaker_alpha_map:
        alpha_vals = list(speaker_alpha_map.values())
        print(f"\n  [VTLN] Distribusi alpha: min={min(alpha_vals):.2f}, "
              f"max={max(alpha_vals):.2f}, mean={np.mean(alpha_vals):.3f}")

    if fmllr_per_spk:
        det_vals = [float(np.linalg.det(tr.A))
                    for tr in fmllr_per_spk.values()]
        print(f"  [SAT] fMLLR det(A): min={min(det_vals):.4f}, "
              f"max={max(det_vals):.4f}, mean={np.mean(det_vals):.4f}")

    print("\n" + "="*60)
    print(f"  HASIL AKHIR v6")
    print(f"  Train WER : {r_tr.get('wer_corpus','N/A')}%")
    print(f"  Val   WER : {r_vl.get('wer_corpus','N/A')}%  "
          f"(mean: {r_vl.get('wer_mean','N/A')}%)")
    print(f"  VTLN      : {'aktif' if USE_VTLN else 'nonaktif'}")
    print(f"  SAT/fMLLR : {'aktif, ' + str(SAT_ITERS) + ' iterasi' if USE_SAT else 'nonaktif'}")
    print(f"  Total     : {(time.time()-t_start)/60:.1f} menit")
    print("="*60)
    print("\n  Bug yang diperbaiki dari v5 (WER 500% → normal):")
    print("  [FIX-v6-1] _fonem_ke_kata: return [] bukan return [fonem_seq]")
    print("  [FIX-v6-2] dekode: return [] bukan return fonem_seq saat gagal")
    print("  [FIX-v6-3] N_HMM_STATES: 4 → 2 (CTC collapse lebih toleran)")
    print("  [FIX-v6-4] AVG_FRAMES_PER_WORD: 30 → 15 (max_words lebih besar)")
    print("\n  Bug yang diperbaiki dari v4 (tetap berlaku):")
    print("  [FIX-Bug-1] normalisasi() mempertahankan diakritik e-aksen")
    print("  [FIX-Bug-2] Fonem eu/ae ditambahkan ke _G2P_DIGRAF")
    print("  [FIX-Bug-3] Evaluasi menggunakan cache_final")
    print("  [FIX-Bug-4] scipy path: log10 ke np.log")
    print("  [FIX-Bug-5] VTLN warp: f_high dari parameter fungsi")
    print("  [FIX-Bug-6] fMLLR: fonem_labels digunakan untuk filter GMM")
    print("  [FIX-Bug-7] LabelSmoothing: weight hanya pada kelas target")
    print("  [FIX-Bug-8] GMM: WARNING eksplisit saat n_components dikurangi")
    print("  [FIX-Bug-9] VTLN: WARNING jika speaker tidak punya file valid")


if __name__ == "__main__":
    main()


[1/10] Membaca CSV ...
  CSV: 1749 audio | 0 speaker unik

[2/10] Ekstraksi MFCC awal (alpha=1.0) ...
  Ekstrak MFCC dari 1749 file ...
    [200/1749] 12s
    [400/1749] 17s
    [600/1749] 20s
    [800/1749] 21s
    [1000/1749] 22s
    [1200/1749] 24s
    [1400/1749] 26s
    [1600/1749] 31s
  Selesai: 1749 audio
  [INFO] Tidak ada kolom speaker → skip speaker CMVN

[3/10] Split train/val ...
  Train: 1400 | Val: 349

[4/10] Lexicon dan N-gram LM ...
  Lexicon: 5070 kata | LM vocab: 4328

[5/10] GMM awal ...
FASE 1: Iterative monophone GMM

  Iter 1/3 ...
  GMM: 28 model dalam 240.0s
  → 28 model monophone

  Iter 2/3 ...
  GMM: 29 model dalam 185.9s
  → 29 model monophone

  Iter 3/3 ...
    [ 1] SIL :    435 frame, K=8, iter=29, conv=True, LL=-45.516
    [ 2] a   : 108866 frame, K=8, iter=23, conv=True, LL=-50.891
    [ 3] ae  :    292 frame, K=8, iter=23, conv=True, LL=-40.84
    [ 4] b   :  18887 frame, K=8, iter=83, conv=True, LL=-46.84
    [ 5] c   :  10291 frame, K=8, iter=41, c